# Exercise: Implementing Text Generation

Welcome! In this exercise, you will implement the core logic for generating text from a trained GPT model. This process, also known as sampling or inference, is what allows a language model to produce creative and coherent text based on a prompt.

By completing this notebook, you will:
*   Implement the single-step prediction logic to get the next token.
*   Write the main autoregressive loop that generates a sequence of tokens.
*   Combine these pieces to generate text from a starting prompt.

Let's get started!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Helper Code (Students should NOT modify this) ---

class MockTokenizer:
    """A simple, deterministic tokenizer for testing."""
    def __init__(self):
        self.vocab = ['a', 'b', 'c', 'd', 'e', '<EOS>']
        self.vocab_size = len(self.vocab)
        self.stoi = {ch: i for i, ch in enumerate(self.vocab)}
        self.itos = {i: ch for i, ch in enumerate(self.vocab)}
        self.eos_token_id = self.stoi['<EOS>']

    def encode(self, s: str) -> torch.Tensor:
        """Converts a string to a tensor of token IDs."""
        ids = [self.stoi[ch] for ch in s]
        return torch.tensor(ids, dtype=torch.long).unsqueeze(0) # (1, T)

    def decode(self, l: list[int]) -> str:
        """Converts a list of token IDs to a string."""
        return "".join([self.itos[i] for i in l])

class MockMiniGPTModel(nn.Module):
    """
    A mock GPT model for deterministic testing.
    This model's logic is simple: it predicts the next token in the vocabulary sequence.
    If the last input token is 'a' (ID 0), it will predict 'b' (ID 1).
    If the last input token is 'e' (ID 4), it will predict '<EOS>' (ID 5).
    """
    def __init__(self, vocab_size):
        super().__init__()
        self.vocab_size = vocab_size

    def forward(self, idx):
        """
        Takes a tensor of token indices `idx` (Batch, Time) and returns
        logits for the next token.
        """
        B, T = idx.shape
        # Create zero logits for the entire sequence
        logits = torch.zeros(B, T, self.vocab_size)
        
        # Get the last token for each sequence in the batch
        last_token = idx[:, -1]
        
        # Predict the next token in the sequence, wrapping around
        next_token_prediction = (last_token + 1) % self.vocab_size
        
        # Set a very high logit for the predicted next token at the last time step
        # This makes the model's prediction deterministic for our tests
        for i in range(B):
            logits[i, -1, next_token_prediction[i]] = 10.0
            
        return logits, None # (logits, loss)

# Instantiate our helpers for the exercises
tokenizer = MockTokenizer()
mock_model = MockMiniGPTModel(tokenizer.vocab_size)
print("Setup complete. You can now proceed to the exercises.")

### Exercise 1: Get the Next Token

Your first task is to implement the logic for generating a *single* next token. This is the core building block of the autoregressive generation loop.

You will write a function `get_next_token` that takes the current sequence of token indices (`idx`) and the model, and performs the following steps:
1.  Get the model's predictions (logits) for the input sequence.
2.  Focus only on the logits for the very last time step, as that's where the prediction for the *next* token lies.
3.  Apply the `softmax` function to these logits to convert them into probabilities.
4.  Sample from the probability distribution to get the index of the next token. `torch.multinomial` is perfect for this.

**Hint:** When you get the logits from the model, they will have the shape `(Batch, Time, VocabSize)`. You are only interested in the predictions that follow the last token in the input, so you'll need to select `logits[:, -1, :]`.

In [ ]:
def get_next_token(idx: torch.Tensor, model: MockMiniGPTModel) -> torch.Tensor:
    """
    Samples the next token from the model's predictions.

    Args:
        idx (torch.Tensor): The current sequence of token indices (B, T).
        model (MockMiniGPTModel): The GPT model.

    Returns:
        torch.Tensor: A tensor containing the predicted next token index (B, 1).
    """
    # Set the model to evaluation mode and disable gradient calculations
    model.eval()
    with torch.no_grad():
        model (MockMiniGPTModel): The GPT model.

    Returns:
        torch.Tensor: A tensor containing the predicted next token index (B, 1).
    """
    # Set the model to evaluation mode and disable gradient calculations
    model.eval()
    with torch.no_grad():
        ### START CODE HERE ###
        # 1. Get logits from the model
        logits, _ = model(idx)
        
        # 2. Get the logits for the last time step
        last_logits = logits[:, -1, :] # Shape: (B, vocab_size)
        
        # 3. Apply softmax to convert logits to probabilities
        probs = F.softmax(last_logits, dim=-1) # Shape: (B, vocab_size)
        
        # 4. Sample the next token from the probability distribution
        idx_next = torch.multinomial(probs, num_samples=1) # Shape: (B, 1)
        ### END CODE HERE ###
        
    return idx_next
        
    return idx_next

In [ ]:
# --- Test Cell for Exercise 1 ---
torch.manual_seed(42) # Set seed for reproducibility of torch.multinomial

# Test case 1: Starting with token 'a' (ID 0)
prompt_idx = torch.tensor([[tokenizer.stoi['a']]], dtype=torch.long)
next_token = get_next_token(prompt_idx, mock_model)
expected_token_id = tokenizer.stoi['b']
assert next_token.item() == expected_token_id, f"Test 1 Failed: Expected next token for 'a' to be 'b' (ID {expected_token_id}), but got ID {next_token.item()}"
print("✅ Test 1 Passed!")

# Test case 2: Starting with 'a', 'b', 'c' (IDs 0, 1, 2)
prompt_idx = torch.tensor([[tokenizer.stoi['a'], tokenizer.stoi['b'], tokenizer.stoi['c']]], dtype=torch.long)
next_token = get_next_token(prompt_idx, mock_model)
expected_token_id = tokenizer.stoi['d']
assert next_token.item() == expected_token_id, f"Test 2 Failed: Expected next token for '...c' to be 'd' (ID {expected_token_id}), but got ID {next_token.item()}"
print("✅ Test 2 Passed!")

# Test case 3: Nearing the end of the vocabulary
prompt_idx = torch.tensor([[tokenizer.stoi['e']]], dtype=torch.long)
next_token = get_next_token(prompt_idx, mock_model)
expected_token_id = tokenizer.eos_token_id
assert next_token.item() == expected_token_id, f"Test 3 Failed: Expected next token for 'e' to be '<EOS>' (ID {expected_token_id}), but got ID {next_token.item()}"
print("✅ Test 3 Passed!")

print("\nAll tests for `get_next_token` passed! Great job!")

### Exercise 2: The Generation Loop

Fantastic! Now that you can predict one token at a time, you'll implement the full autoregressive loop in the `generate` function.

This function will repeatedly generate tokens and append them to the sequence until one of two conditions is met:
1.  The desired number of new tokens (`max_new_tokens`) has been generated.
2.  The model generates a special end-of-sequence (`end_of_sequence_token`) token.

Inside the loop, you will:
1.  Call the model to get the next token's probabilities (this is the logic you just wrote in Exercise 1).
2.  Sample the next token.
3.  Append the new token to your running sequence `idx`.
4.  Check if the newly generated token is the `end_of_sequence_token`. If it is, stop the loop.

This process is "autoregressive" because the model's output at one step becomes part of its input for the next step.

In [ ]:
def generate(idx: torch.Tensor, model: MockMiniGPTModel, max_new_tokens: int, end_of_sequence_token: int) -> torch.Tensor:
    """
    Generates a sequence of tokens starting from a prompt `idx`.

    Args:
        idx (torch.Tensor): The prompt sequence of token indices (B, T).
        model (MockMiniGPTModel): The GPT model.
        max_new_tokens (int): The maximum number of new tokens to generate.
        end_of_sequence_token (int): The token ID that signals the end of a sequence.

    Returns:
        torch.Tensor: The generated sequence of token indices, including the prompt (B, T_new).
    """
    # Set the model to evaluation mode
    model.eval()
    
    # Loop for the specified number of tokens
    for _ in range(max_new_tokens):
        with torch.no_grad():
        model (MockMiniGPTModel): The GPT model.
        max_new_tokens (int): The maximum number of new tokens to generate.
        end_of_sequence_token (int): The token ID that signals the end of a sequence.

    Returns:
        torch.Tensor: The generated sequence of token indices, including the prompt (B, T_new).
    """
    # Set the model to evaluation mode
    model.eval()
    
    # Loop for the specified number of tokens
    for _ in range(max_new_tokens):
        with torch.no_grad():
            ### START CODE HERE ###
            # 1. Get the logits from the model
            logits, _ = model(idx)
            
            # 2. Focus on the last time step
            last_logits = logits[:, -1, :]
            
            # 3. Apply softmax to get probabilities
            probs = F.softmax(last_logits, dim=-1)
            
            # 4. Sample the next token
            idx_next = torch.multinomial(probs, num_samples=1)
            
            # 5. Check for end of sequence token
            if idx_next.item() == end_of_sequence_token:
                break
                
            # 6. Append the sampled token to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)
            ### END CODE HERE ###
            
    return idx
            
    return idx

In [ ]:
# --- Test Cell for Exercise 2 ---
torch.manual_seed(42) # Ensure consistent sampling

# Test case 1: Generate 4 new tokens
prompt_idx = tokenizer.encode('a') # Starts with ID 0
max_tokens = 4
eos_token = tokenizer.eos_token_id

generated_idx = generate(prompt_idx, mock_model, max_tokens, eos_token)
expected_sequence = tokenizer.encode('abcde').tolist()[0] # 0, 1, 2, 3, 4
actual_sequence = generated_idx.tolist()[0]

assert actual_sequence == expected_sequence, f"Test 1 Failed: Expected sequence {expected_sequence}, but got {actual_sequence}"
print("✅ Test 1 Passed! (Correct length and sequence)")

# Test case 2: Stop generation at <EOS> token
prompt_idx = tokenizer.encode('d') # Starts with ID 3
max_tokens = 10 # More than enough to reach <EOS>
eos_token = tokenizer.eos_token_id

generated_idx = generate(prompt_idx, mock_model, max_tokens, eos_token)
expected_sequence = tokenizer.encode('de').tolist()[0] # 3, 4. Should stop before generating <EOS>
actual_sequence = generated_idx.tolist()[0]

# The loop should break *after* 'e' is generated, which predicts <EOS> next. So the final sequence is 'de'
assert actual_sequence == expected_sequence, f"Test 2 Failed: Expected sequence {expected_sequence} (stopped by EOS), but got {actual_sequence}"
print("✅ Test 2 Passed! (Correctly stopped at <EOS>)")

# Test case 3: Start with a longer prompt
prompt_idx = tokenizer.encode('ab') # Starts with IDs 0, 1
max_tokens = 2
eos_token = tokenizer.eos_token_id

generated_idx = generate(prompt_idx, mock_model, max_tokens, eos_token)
expected_sequence = tokenizer.encode('abcd').tolist()[0] # 0, 1, 2, 3
actual_sequence = generated_idx.tolist()[0]

assert actual_sequence == expected_sequence, f"Test 3 Failed: Expected sequence {expected_sequence}, but got {actual_sequence}"
print("✅ Test 3 Passed! (Handled longer prompt)")

print("\nAll tests for `generate` passed! You've successfully implemented autoregressive text generation!")

### (Optional) Challenge: Top-k Sampling

The sampling method you implemented (`torch.multinomial`) considers all words in the vocabulary. Sometimes, this can lead to strange or nonsensical word choices if the model assigns a small but non-zero probability to them.

A popular technique to mitigate this is **Top-k Sampling**. The idea is simple:
1.  Get the logits from the model.
2.  Instead of using all logits, keep only the `k` highest logits. Set all other logits to a very small number (like negative infinity).
3.  Apply softmax and sample from this modified, smaller set of potential tokens.

This prevents the model from picking highly unlikely tokens, often leading to more coherent text.

**Your challenge:** Modify your `generate` function to include an optional `top_k` argument. If `top_k` is provided, implement the top-k filtering logic before the softmax and sampling steps.

You may find `torch.topk` useful for this!

In [ ]:
# --- Integration Test: Putting It All Together ---

# Now let's see your code in action! We'll use your `generate` function to
# create a sequence and then use the tokenizer to decode it back into
# human-readable text.

torch.manual_seed(1337) # A different seed for a different result
print("Running integration test...")

# 1. Define the prompt
prompt_string = "a"
print(f"Prompt: '{prompt_string}'")

# 2. Encode the prompt into token IDs
prompt_idx = tokenizer.encode(prompt_string)

# 3. Generate new tokens using your function
generated_idx = generate(
    idx=prompt_idx,
    model=mock_model,
    max_new_tokens=10, # Ask for 10 tokens
    end_of_sequence_token=tokenizer.eos_token_id
)

# 4. Decode the result
generated_list = generated_idx.squeeze().tolist()
generated_text = tokenizer.decode(generated_list)
print(f"Generated Text: '{generated_text}'")

# The mock model should produce 'abcde' and then stop because the next predicted
# token is <EOS>.
expected_output = "abcde"
print(f"Expected Text:  '{expected_output}'")
assert generated_text == expected_output, f"Final test failed! Expected '{expected_output}', but got '{generated_text}'."

print("\nCongratulations! You have successfully implemented and tested the text generation loop.")
print("This is the fundamental process that powers large language models like GPT.")